# Notebook 7: Representation Engineering & Steering Vectors

Probing (nb06) showed us how to **read** information from model activations. Now we go further: we can **write** to activations to control model behavior.

The core idea: if a concept (e.g., honesty, happiness, refusal) corresponds to a direction in activation space, you can add or subtract that direction during inference to steer the model. Two main approaches:
1. **Activation addition** ([Turner et al., 2023](https://arxiv.org/abs/2308.10248)): Add a steering vector to the residual stream
2. **Representation Engineering (RepE)** ([Zou et al., 2023](https://arxiv.org/abs/2310.01405)): More systematic framework for finding and applying control vectors

## Section 2: Finding a Steering Vector via Contrastive Pairs

The simplest way to find a steering vector: collect activations for positive and negative examples, then take the difference of means.

For example, to find an "honesty" direction:
- **Positive**: model activations when generating honest responses
- **Negative**: model activations when generating dishonest responses
- **Steering vector**: mean(positive) - mean(negative)

Let's build a **sentiment** steering vector as a concrete example.

> **Note:** The original RepE paper (Zou et al., 2023) extracts concept directions using PCA on contrastive activations (specifically, the first principal component of the difference between positive and negative activation sets). The difference-of-means approach shown here is a simpler alternative that works well in practice and is easier to reason about.


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from transformer_lens import HookedTransformer

model = HookedTransformer.from_pretrained("gpt2-small")
model.eval()

# Contrastive pairs for sentiment (positive vs negative)
positive_prompts = [
    "I feel absolutely wonderful today because",
    "This is the most amazing experience of",
    "I love how beautiful and perfect this",
    "Everything is going brilliantly and I am",
    "The happiest moment of my life was",
    "I am so grateful and thankful for",
    "This brings me incredible joy and",
    "What a fantastic and delightful surprise",
    "I am thrilled and excited about the",
    "The world is full of beauty and",
]

negative_prompts = [
    "I feel absolutely terrible today because",
    "This is the most awful experience of",
    "I hate how ugly and horrible this",
    "Everything is going horribly and I am",
    "The worst moment of my life was",
    "I am so angry and frustrated with",
    "This brings me incredible pain and",
    "What a terrible and dreadful disaster",
    "I am horrified and disgusted by the",
    "The world is full of suffering and",
]

# Collect activations at a middle layer
target_layer = 6

def get_mean_activation(prompts, layer):
    """Get mean activation at last token across prompts."""
    acts = []
    for prompt in prompts:
        _, cache = model.run_with_cache(prompt)
        act = cache[f"blocks.{layer}.hook_resid_post"][0, -1, :].detach()
        acts.append(act)
    return torch.stack(acts).mean(dim=0)

pos_mean = get_mean_activation(positive_prompts, target_layer)
neg_mean = get_mean_activation(negative_prompts, target_layer)

# Steering vector = positive - negative direction
steering_vector = pos_mean - neg_mean
steering_vector = steering_vector / steering_vector.norm()  # Normalize

print(f"Steering vector computed (layer {target_layer})")
print(f"Vector norm before normalization: {(pos_mean - neg_mean).norm():.2f}")

## Section 3: Applying the Steering Vector

Now we add (or subtract) scaled versions of the steering vector during generation to shift the model's outputs.

- **Positive alpha**: push toward positive sentiment
- **Negative alpha**: push toward negative sentiment
- **Zero alpha**: baseline (no intervention)

In [ ]:
def generate_with_steering(prompt, steering_vec, layer, alpha=0.0, max_tokens=30):
    """Generate text with a steering vector added at the specified layer."""
    
    def steering_hook(activation, hook):
        # NOTE: This adds the steering vector to ALL token positions in the sequence.
        # A more targeted approach would steer only at specific positions (e.g., the
        # last token), which can sometimes give cleaner results.
        activation[:, :, :] += alpha * steering_vec.to(activation.device)
        return activation
    
    tokens = model.to_tokens(prompt)
    
    for _ in range(max_tokens):
        if alpha != 0:
            logits = model.run_with_hooks(
                tokens,
                fwd_hooks=[(f"blocks.{layer}.hook_resid_post", steering_hook)]
            )
        else:
            logits = model(tokens)
        
        # Greedy sampling
        next_token = logits[0, -1, :].argmax().unsqueeze(0).unsqueeze(0)
        tokens = torch.cat([tokens, next_token], dim=1)
        
        # Stop at period or newline
        if next_token.item() in [model.to_single_token("."), model.to_single_token("\n")]:
            break
    
    return model.to_string(tokens[0])

# Test with a neutral prompt
test_prompt = "I went to the store and"
print("=" * 70)
print(f"Prompt: '{test_prompt}'")
print("=" * 70)

for alpha in [-10, -5, 0, 5, 10]:
    output = generate_with_steering(test_prompt, steering_vector, target_layer, alpha=alpha)
    label = "baseline" if alpha == 0 else f"\u03b1={alpha:+d}"
    print(f"\n[{label}]: {output}")

In [ ]:
neutral_prompts = [
    "Today I decided to",
    "The restaurant was",
    "My friend told me that",
    "Looking at the sky, I",
]

print("Steering effect on neutral prompts")
print("=" * 70)

for prompt in neutral_prompts:
    print(f"\nPrompt: '{prompt}'")
    for alpha in [-8, 0, 8]:
        output = generate_with_steering(prompt, steering_vector, target_layer, alpha=alpha)
        label = "negative" if alpha < 0 else ("positive" if alpha > 0 else "baseline")
        # Trim prompt from output for clarity
        generated = output[len(model.to_string(model.to_tokens(prompt)[0])):]
        print(f"  [{label:>8}]: ...{generated}")

## Section 4: Layer Selection and Strength

The choice of which layer to intervene at and how strong the steering should be matters a lot:
- **Too weak** = no effect
- **Too strong** = gibberish
- **Wrong layer** = unexpected behavior

Let's sweep over layers and strengths to see how it affects generation.

In [ ]:
# Test different layers
test_prompt = "The movie was"
alphas = [0, 5, 10]
layers_to_test = [0, 3, 6, 9, 11]

results = {}
for layer in layers_to_test:
    sv = get_mean_activation(positive_prompts, layer) - get_mean_activation(negative_prompts, layer)
    sv = sv / sv.norm()
    
    results[layer] = {}
    for alpha in alphas:
        output = generate_with_steering(test_prompt, sv, layer, alpha=alpha, max_tokens=20)
        generated = output[len(model.to_string(model.to_tokens(test_prompt)[0])):]
        results[layer][alpha] = generated

print(f"Prompt: '{test_prompt}'")
print("=" * 70)
for layer in layers_to_test:
    print(f"\nLayer {layer}:")
    for alpha in alphas:
        label = "base" if alpha == 0 else f"\u03b1={alpha}"
        print(f"  [{label:>5}]: ...{results[layer][alpha]}")

## Section 5: Concept Vectors Beyond Sentiment

Steering vectors can encode many concepts beyond sentiment:
- **Honesty/deception**: Steer toward more truthful outputs
- **Formality**: Control register from casual to formal
- **Refusal**: Make model more/less likely to refuse requests
- **Language**: Steer between languages in multilingual models
- **Safety**: Anthropic found safety-relevant features in Claude's activations

The power here is that you can control behaviors that weren't explicitly trained, by leveraging the model's internal representations.

**Keep in mind:**
- Steering is approximate -- it shifts the distribution, doesn't guarantee specific behavior
- Too strong steering degrades output quality
- The optimal layer and strength vary by concept and model
- Some concepts may not have clean linear representations

---
### Running Example: IOI — Steering the Name

This is part of our **running example** investigating how GPT-2-small handles the Indirect Object Identification (IOI) task across all techniques in this guide.

**The task**: "When Mary and John went to the store, John gave a drink to" — the model should predict "Mary".

Here we compute a "Mary direction" by contrasting prompts where Mary is the correct answer versus where she is not, then steer with it to shift the model's name prediction on IOI-like prompts.

In [ ]:
# Running Example: IOI — Steering the Name with a "Mary direction"
prompt = "When Mary and John went to the store, John gave a drink to"

# Compute a "Mary direction": contrast Mary-correct vs John-correct IOI prompts
mary_prompts = [
    "When Mary and John went to the store, John gave a drink to",
    "When Mary and John walked to the park, John handed a snack to",
    "When Mary and John visited the office, John passed a note to",
    "When Mary and John arrived at the cafe, John offered a cookie to",
    "When Mary and John stopped by the library, John slid a book to",
]
john_prompts = [
    "When John and Mary went to the store, Mary gave a drink to",
    "When John and Mary walked to the park, Mary handed a snack to",
    "When John and Mary visited the office, Mary passed a note to",
    "When John and Mary arrived at the cafe, Mary offered a cookie to",
    "When John and Mary stopped by the library, Mary slid a book to",
]

mary_mean = get_mean_activation(mary_prompts, target_layer)
john_mean = get_mean_activation(john_prompts, target_layer)

name_direction = mary_mean - john_mean
name_direction = name_direction / name_direction.norm()

# Steer a neutral IOI-like prompt and see how the prediction shifts
test = "When Alice and Bob went to the store, Bob gave a gift to"
print(f"Test prompt: '{test}'")
print(f"Steering with 'Mary direction' at layer {target_layer}:\n")

for alpha in [-5, 0, 5]:
    output = generate_with_steering(test, name_direction, target_layer, alpha=alpha, max_tokens=5)
    print(f"  alpha={alpha:+d}: {output}")

## Exercises

### Exercise 1: Build Your Own Steering Vector

Create a "formality" steering vector. Collect activations for a set of contrastive prompts (formal vs. casual sentences). Compute the contrastive mean difference. Apply it during generation and observe the effect on the model's output style.

<details>
<summary>Hint</summary>

`formal_direction = mean(formal_activations) - mean(casual_activations)`. Normalize the vector. Then use `model.run_with_hooks()` with a hook that adds `alpha * formal_direction` to the residual stream at a chosen middle layer (e.g., layer 6). Try positive alpha to push toward formality and negative alpha to push toward casualness.

</details>

In [ ]:
formal = [
    "The committee hereby resolves to adopt the proposal",
    "We respectfully request your consideration of this matter",
    "The aforementioned regulations shall take effect immediately",
    "It is our professional opinion that the project merits approval",
    "The board of directors convened to discuss quarterly performance",
    "Please be advised that the deadline has been extended",
    "We wish to express our sincere gratitude for your cooperation",
    "The undersigned parties agree to the terms set forth herein",
]
casual = [
    "hey so like we decided to go with it",
    "yeah honestly i think its pretty cool",
    "lol that rule starts right now i guess",
    "we totally think this project is worth doing",
    "the team got together to chat about how things are going",
    "just a heads up the deadline got pushed back",
    "thanks a ton for helping out with everything",
    "we're all good with the deal as is",
]

# Collect activations
steer_layer = 6
formal_mean = get_mean_activation(formal, steer_layer)
casual_mean = get_mean_activation(casual, steer_layer)

# Compute formality direction
formality_direction = formal_mean - casual_mean
formality_direction = formality_direction / formality_direction.norm()

# TODO: Try modifying this!
# Generate with and without steering
test_prompt = "The company announced that"
for alpha in [-8, 0, 8]:
    output = generate_with_steering(test_prompt, formality_direction, steer_layer, alpha=alpha)
    label = "casual" if alpha < 0 else ("formal" if alpha > 0 else "baseline")
    print(f"[{label:>8}]: {output}")

### Exercise 2: Steering Strength Sweep

Take a steering vector (e.g., the sentiment vector from Section 2 or the formality vector from Exercise 1) and apply it at different strengths: alpha = -3, -2, -1, 0, 1, 2, 3. For each, generate 5 completions. Observe how behavior changes. Is there a "sweet spot"? What happens at extreme values?

<details>
<summary>Hint</summary>

Use `model.run_with_hooks()` with a hook that adds `alpha * direction` to the residual stream at a chosen layer. At alpha=0 you get the baseline. Moderate values should produce coherent but shifted text. Extreme values (large |alpha|) often degrade into gibberish -- finding the boundary is the goal.

</details>

In [ ]:
alphas = [-3, -2, -1, 0, 1, 2, 3]
test_prompt = "I think the best thing about this is"
steer_layer = 6  # Use the layer from your steering vector

# TODO: Try modifying this!
# Use the sentiment steering_vector from Section 2
for alpha in alphas:
    print(f"\n--- alpha = {alpha:+d} ---")
    for i in range(5):
        output = generate_with_steering(test_prompt, steering_vector, steer_layer, alpha=alpha)
        generated = output[len(model.to_string(model.to_tokens(test_prompt)[0])):]
        print(f"  {i+1}: ...{generated}")

# Analyze the pattern:
# - At what alpha does the text become noticeably shifted?
# - At what alpha does coherence break down?
# - Is the effect symmetric for positive and negative alpha?

## Section 6: Key Takeaways & Further Reading

**What you should remember:**
- Representation engineering turns interpretability into *control*
- Steering vectors = contrastive mean difference of activations
- Adding/subtracting from the residual stream shifts model behavior predictably
- Layer choice and steering strength are key hyperparameters
- This works for many semantic concepts beyond sentiment

**Connection to safety:**
- Steering vectors for honesty/refusal could serve as lightweight safety interventions
- Anthropic's work on extracting safety-relevant features connects directly here
- Understanding which directions control which behaviors is crucial for alignment

**Further reading:**
- [Representation Engineering](https://arxiv.org/abs/2310.01405) (Zou et al., 2023)
- [Activation Addition](https://arxiv.org/abs/2308.10248) (Turner et al., 2023)
- [Steering GPT-2-XL by adding an activation vector](https://www.alignmentforum.org/posts/5spBue2z2tw4JuDCx) (Turner et al., blog post)
- [Concept Bottleneck Large Language Models](https://arxiv.org/abs/2412.07992) (2024)

**Next**: Notebook 08 -- Circuit Tracing (Anthropic's 2025 attribution graph methodology)